In [ ]:
from datasets import load_dataset
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
dataset = load_dataset("dair-ai/emotion")
train_data = dataset["train"]
test_data = dataset["test"]
# use gpu or cpu and print it
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Label Mapping and Conversion Function

In [ ]:
id2label = {
    0: "sadness",
    1: "joy",
    2: "love",
    3: "anger",
    4: "fear",
    5: "surprise"
}
def convert_label(label):
    return id2label[label]

Tokenization Function

In [ ]:
def tokenize(text):
    text = text.lower().split()
    return text

Vocabulary Construction and Word Frequency Counting

In [ ]:
from collections import Counter
counter = Counter()
for i in train_data:
    counter.update(tokenize(i["text"]))

vocab = {word: i+2 for i , (word,_) in enumerate(counter.most_common(20000))}
vocab["<pad>"] = 0
vocab["<unk>"] = 1

Text Encoding: Token-to-ID Mapping and Tensor Conversion

In [ ]:
def encode(text):
    tokens = tokenize(text)
    ids = [vocab.get(t,vocab["<unk>"] ) for t in tokens]
    return torch.tensor(ids ,device=device)

Batch Collation: Padding, Tensor Assembly, and DataLoader Construction

In [ ]:
def collate_fn(batch):
    texts = [encode(item["text"]) for item in batch]
    labels = torch.tensor([item["label"] for item in batch] , device=device)
    texts = nn.utils.rnn.pad_sequence(texts , batch_first=True)
    texts = texts.to(device)
    return texts , labels

train_loader = DataLoader(train_data,batch_size=64,shuffle=True,collate_fn=collate_fn)
test_loader = DataLoader(test_data,batch_size=64,shuffle=False,collate_fn=collate_fn)

LSTM-Based Emotion Classifier: Embedding, Sequence Modeling, and Final Prediction Layer

In [ ]:
class LSTMModel(nn.Module):
    def __init__(self,vocab_size,embed_dim,hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size,embed_dim)
        self.lstm = nn.LSTM(embed_dim,hidden_dim,batch_first=True)
        self.fc = nn.Linear(hidden_dim,6)
    def forward(self,x):
        embedded = self.embedding(x)
        _,(hidden,cell) = self.lstm(embedded)
        hidden = hidden.squeeze(0)
        return self.fc(hidden)

Model Initialization: LSTM Configuration, Loss Function Setup, and Optimizer Definition

In [ ]:
model = LSTMModel(len(vocab) , embed_dim=256 , hidden_dim= 256).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(),lr=0.001)

Training Loop: Forward Pass, Loss Computation, Backpropagation, and Epoch-Level Reporting

In [ ]:
for epoch in range (14):
    total_loss = 0
    for texts,labels in train_loader:
        texts , labels = texts.to(device),labels.to(device)
        optimizer.zero_grad()
        outputs = model(texts)
        loss = criterion(outputs,labels)
        loss.backward()
        optimizer.step()
        total_loss+=loss.item()
    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1}, Avg Loss: {avg_loss:.4f}")

Inference Function: Text Encoding, Model Evaluation, and Label Decoding

In [ ]:
def predict(text):
    model.eval()
    with torch.no_grad():
        ids = encode(text).unsqueeze(0).to(device)
        output = model(ids)
        pred = torch.argmax(output).item()
        return id2label[pred]

Inference Testing: Sample Predictions for Multiple Input Sentences

In [ ]:
print(predict("I am very happy today"))
print(predict("I feel so lonely"))
print(predict("I love this moment"))
print(predict("This is terrifying"))
print(predict("I am extremely angry"))
print(predict("Wow, that's surprising"))